# Trabajo Práctico Aprendizaje Automático 1

In [2]:
import pandas as pd
import numpy as np


Fijamos una semilla para tener reproducibilidad en los resultados

In [20]:
SEED = 42
rng = np.random.default_rng(SEED)

## 1. Separación de datos

Primero que nada guardamos los datos que vamos a usar en el trabajo.

In [17]:
data = pd.read_csv('../Data/data.csv')

Luego lo que debemos hacer antes de empezar con el análisis exploratorio y el desarrollo de modelos es, dividir nuestros datos en desarrollo y control. Esto se hace con el objetivo de que al final del trabajo podamos reportar la performance esperada del modelo final con datos de la vida real.

Para hacer esto se me ocurrieron dos ideas (nada innovador) -> mis 2 huevos vas a innovar (firma giannisluca a sebasouto):

1. Mezclar todos los datos y quedarnos con el porcentaje deseado.

2. Mezclar todos los datos y quedarnos con el porcentaje deseado pero estratificando. Nose si hacer esto es trampa, porque haciendo eso ya voy a conocer algo de los datos de control. 

Voy a implementar ambos y despues decidimos. IMPORTANTE: antes de seguir con los otros puntos elegir una forma de dividir y no volver a tocar esos datos.

In [18]:
#Defino el porcentaje que vamos a usar para control. A debatir.. me pareció mucho usar el 20%. 

porcent = 0.10

### Opción 1:

In [21]:
# Mezclamos los datos directamente del dataframe. 
data = data.sample(frac=1, random_state=SEED).reset_index(drop=True)

#Tomamos el porcentaje de control y desarrollo
n_control = int(len(data)*porcent)

#Dividimos..
data_control = data.iloc[:n_control].reset_index(drop=True)
data_dev = data.iloc[n_control:].reset_index(drop=True)

### Opción 2:

In [22]:
#Primero obtenemos los índices de cada clase en el data frame, ya que queremos estratificar. Tuve que agregar los .copy() porque me tiraba error.
indices_pos = data[data['target'] == 1].index.to_numpy().copy()
indices_neg = data[data['target'] == 0].index.to_numpy().copy()

# Luego mezclamos los índices de cada clase. 
rng.shuffle(indices_pos)
rng.shuffle(indices_neg)

# Calculamos la cantidad de instancias de control. ACA es la parte donde se estratifica.
n_control_pos = int(np.round(len(indices_pos) * porcent))
n_control_neg = int(np.round(len(indices_neg) * porcent))

# Partimos los indices de cada clase en control y desarrollo.
idx_control_pos = indices_pos[:n_control_pos]
idx_dev_pos     = indices_pos[n_control_pos:]

idx_control_neg = indices_neg[:n_control_neg]
idx_dev_neg     = indices_neg[n_control_neg:]

#Concatenamos los indices de control y desarrollo de cada clase para obtener los índices finales.
idx_control = np.concatenate([idx_control_pos, idx_control_neg])
idx_dev     = np.concatenate([idx_dev_pos, idx_dev_neg])

# Y mezclamos para que no nos queden los positivos por un lado y los negativos por el otro. Esto nose si es al pedo
rng.shuffle(idx_control)
rng.shuffle(idx_dev)

# Finalmente construimos los dataframes de desarrollo y control 
data_dev = data.loc[idx_dev].reset_index(drop=True)
data_control = data.loc[idx_control].reset_index(drop=True)

# Verificación de la estratificación
print(f"Total de datos: {len(data)}")
print(f"Desarrollo: {len(data_dev)} filas | Proporción Positivos: {data_dev['target'].mean():.4f}")
print(f"Control:    {len(data_control)} filas  | Proporción Positivos: {data_control['target'].mean():.4f}")

Total de datos: 500
Desarrollo: 450 filas | Proporción Positivos: 0.2822
Control:    50 filas  | Proporción Positivos: 0.2800


## 2: Construcción de modelos

### 2.1: Entrenar arbol de desición

Primero, armamos los train con la data ya separada por Desarrollo y Control:

In [23]:
X_train = data_dev.drop(columns=['target'])
y_train = data_dev['target']

Entrenamos un arbol de desición con altura maxima 3 e hiperparam por defecto

In [24]:
from sklearn.tree import DecisionTreeClassifier

In [9]:
#este modelo por ahora no se usa para nada, es la base
mode_full_train = DecisionTreeClassifier(max_depth=3)
mode_full_train.fit(X_train, y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",3
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples a

### 2.2: Evaluar K-Fold

Hago folds estratificados por desbalance

In [9]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
)



In [11]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=912)#comoteduelelaco...

kf por dentro tiene indexaciones por cada fold. Es decir, se ve algo asi:

fold 1: ((0,1,2,5),(3,4)) -> entreno con train[0] , train[1] ,..., y evaluo con 3,4. (escribir esto lindo)

In [25]:


fold = 1 #negrada, puede quedar mejor esto si se itera con un for pero queda bastante mas dificil de leer.
         #lo voy aumentando por cada for (es para printear boludeces nomas)
         #TODO: borrar la parte que dice -negrada- para el proximo que lea esto!

#resultados fold quedan en este result_folds en orden, no es muy lindo. TODO: si se les ocurre como ponerlo mas lindo cambienlo
results_folds = []

for train_idx, test_idx in kf.split(X_train, y_train): #train idx: indices para entrenar fold i, test idx: indices para testear fold i


    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[test_idx] #agarro features train y val para el fold i

    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[test_idx] #agarro labels train y val para el fold i

    #lo sig como chequeo, not nescesario
    
    print(f"Cantidad de datos en el fold {fold} de train: {len(y_train_fold)}")
    print(f"Cantidad de datos en el fold {fold} de validation: {len(y_val_fold)}")
    print(f"Proporcion de datos positivos en el fold {fold} de train: {y_train_fold.mean():.4f}")
    print(f"Proporcion de datos positivos en el fold {fold} de validation: {y_val_fold.mean():.4f}")


    model = DecisionTreeClassifier(max_depth=3)
    model.fit(X_train_fold, y_train_fold) #entreno modelo en datos fold


    y_pred_train_fold = model.predict(X_train_fold) #predicciones del modelo entrenado en los datos de train fold
    y_pred_val_fold = model.predict(X_val_fold) #predicciones del modelo entrenado en los datos de validación fold

    y_prob_train_fold = model.predict_proba(X_train_fold)[:, 1] #probabilidades de la clase positiva para los datos de train fold
    y_prob_val_fold = model.predict_proba(X_val_fold)[:, 1] #probabilidades de la clase positiva para los datos de validación fold

    results_folds.append({
        "Accuracy train": accuracy_score(y_train_fold, y_pred_train_fold),
        "Accuracy validation": accuracy_score(y_val_fold, y_pred_val_fold),
        "AUPRC train": average_precision_score(y_train_fold, y_prob_train_fold),
        "AUPRC validation": average_precision_score(y_val_fold, y_prob_val_fold),
        "AUCROC train": roc_auc_score(y_train_fold, y_prob_train_fold),
        "AUCROC validation": roc_auc_score(y_val_fold, y_prob_val_fold),
    })

    fold += 1


#le puse de nombres validación, no se si es ese o test, creo q en la practica lo habiamos llamado val
#same shi




Cantidad de datos en el fold 1 de train: 360
Cantidad de datos en el fold 1 de validation: 90
Proporcion de datos positivos en el fold 1 de train: 0.2806
Proporcion de datos positivos en el fold 1 de validation: 0.2889
Cantidad de datos en el fold 2 de train: 360
Cantidad de datos en el fold 2 de validation: 90
Proporcion de datos positivos en el fold 2 de train: 0.2806
Proporcion de datos positivos en el fold 2 de validation: 0.2889
Cantidad de datos en el fold 3 de train: 360
Cantidad de datos en el fold 3 de validation: 90
Proporcion de datos positivos en el fold 3 de train: 0.2833
Proporcion de datos positivos en el fold 3 de validation: 0.2778
Cantidad de datos en el fold 4 de train: 360
Cantidad de datos en el fold 4 de validation: 90
Proporcion de datos positivos en el fold 4 de train: 0.2833
Proporcion de datos positivos en el fold 4 de validation: 0.2778
Cantidad de datos en el fold 5 de train: 360
Cantidad de datos en el fold 5 de validation: 90
Proporcion de datos positivos 

In [20]:
results_df = pd.DataFrame(results_folds)

#esto de aca abajo no lo tengo clarisimo tengo sueño mañana lo veo bien, es solo para ver resultados ahora
results_df.index = range(1, len(results_df) + 1)
results_df.index.name = "Fold"

display(results_df.round(3))

,Accuracy train,Accuracy validation,AUPRC train,AUPRC validation,AUCROC train,AUCROC validation
Fold,,,,,,
1,0.803,0.689,0.608,0.321,0.803,0.564
2,0.831,0.711,0.593,0.346,0.733,0.594
3,0.803,0.778,0.637,0.463,0.841,0.700
4,0.833,0.778,0.667,0.507,0.820,0.716
5,0.850,0.722,0.653,0.378,0.789,0.597


In [19]:
#agrego promedios
results_df.loc["Promedio"] = results_df.mean()
display(results_df.round(3))

,Accuracy train,Accuracy validation,AUPRC train,AUPRC validation,AUCROC train,AUCROC validation
Fold,,,,,,
1,0.803,0.689,0.608,0.321,0.803,0.564
2,0.831,0.711,0.593,0.346,0.733,0.594
3,0.803,0.778,0.637,0.463,0.841,0.700
4,0.833,0.778,0.667,0.507,0.820,0.716
5,0.850,0.722,0.653,0.378,0.789,0.597
Promedio,0.824,0.736,0.631,0.403,0.797,0.634


In [ ]:
#agrgo global solo de val
results_df.loc["Global"] =

In [ ]:
#pd (postdata): esto nos deja un indexacion mixta para este df: numeros del 1 al 5 (folds) y string "Promedio"!
#TODO: agregar score final con lo visto en clase

Faltan los scores finales, hay q ver q vimos en clase no me acuerdo. Los meto despues de leer las teos.

### 2.3 Evaluar configuraciones

In [4]:
from sklearn.model_selection import ParameterGrid

Pongo parametros a usar en una grilla para usar ParameterGrid

In [5]:
parametros = {
    "max_depth": [3, 5, None],
    "criterion": ["gini", "entropy"]
}


In [27]:
pg = list(ParameterGrid(parametros))


Hago algo parecido al punto 2.2

In [35]:
resultados = [] # para guardar los resultados

for param in pg: #uso los parametros guardados en pg

    acc_train_folds = []
    acc_val_folds = []

    for train_idx, val_idx in kf.split(X_train, y_train): #evalua config en los 5 folds

        X_train_fold = X_train.iloc[train_idx]      
        X_val_fold = X_train.iloc[val_idx]         

        y_train_fold = y_train.iloc[train_idx]
        y_val_fold = y_train.iloc[val_idx]

        model = DecisionTreeClassifier(**param) # hago **param asi no tengo que escribir los 6 arboles manualmente

        model.fit(X_train_fold, y_train_fold)

        y_pred_train = model.predict(X_train_fold)
        y_pred_val = model.predict(X_val_fold)

        acc_train_folds.append(                             # guardo accuracy sobre el conj de entrenamiento de esta iteracion
            accuracy_score(y_train_fold, y_pred_train)
        )

        acc_val_folds.append(                                  # guardo accuracy sobre el fold de validacion de esta iteracion
            accuracy_score(y_val_fold, y_pred_val)
        )

    resultados.append({                         # guardo los resultados
        "max_depth": param["max_depth"],
        "criterion": param["criterion"],
        "Accuracy train": np.mean(acc_train_folds),             # calculo el promedio de la accuracy de train 
        "Accuracy validation": np.mean(acc_val_folds)           # calculo el promedio de la accuracy de val
    })

Despues veo los resultados

In [34]:
resultados_df = pd.DataFrame(resultados)

resultados_df

,max_depth,criterion,Accuracy train,Accuracy validation
0,3.0,gini,0.823889,0.735556
1,5.0,gini,0.897222,0.697778
2,NaN,gini,1.000000,0.682222
3,3.0,entropy,0.780556,0.675556
4,5.0,entropy,0.873889,0.700000
5,NaN,entropy,1.000000,0.646667
